# SciPy: Optimization, Integration & ODEs

## Overview
SciPy builds on NumPy and provides:
- Optimization algorithms (gradient-based and derivative-free)
- Numerical integration
- ODE solvers
- Interpolation
- Signal processing

## Why SciPy for Engineering?
- CFD post-processing requires integration and interpolation
- Optimization is central to design and control
- ODE solvers model dynamic systems

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize, integrate, interpolate

# 1. Optimization
print("=== Optimization ===")

# Minimize a function
def rosenbrock(x):
    """Rosenbrock function (classic test problem)"""
    return sum(100.0 * (x[1:] - x[:-1]**2.0)**2.0 + (1 - x[:-1])**2.0)

x0 = np.array([0, 0, 0])
result = optimize.minimize(rosenbrock, x0, method='Nelder-Mead')

print(f"Rosenbrock minimum:")
print(f"x* = {result.x}")
print(f"f(x*) = {result.fun:.6f}")
print(f"Iterations: {result.nit}")

# Constrained optimization
def objective(x):
    return x[0]**2 + x[1]**2

constraints = [{'type': 'ineq', 'fun': lambda x: x[0] + x[1] - 1}]
bounds = [(0, None), (0, None)]

result = optimize.minimize(objective, [0.5, 0.5], method='SLSQP',
                           bounds=bounds, constraints=constraints)

print(f"\nConstrained optimization:")
print(f"x* = {result.x}")
print(f"f(x*) = {result.fun:.6f}")

## Numerical Integration

Essential for computing integrals of CFD fields (e.g., drag, lift, heat flux).

In [ ]:
# 2. Numerical Integration
print("\n=== Numerical Integration ===")

# 1D integration
f = lambda x: np.sin(x) * np.exp(-x/10)
result, error = integrate.quad(f, 0, 10)
print(f"∫₀¹⁰ sin(x)·exp(-x/10) dx = {result:.6f} ± {error:.2e}")

# 2D integration
g = lambda x, y: x**2 + y**2
result, error = integrate.dblquad(g, 0, 1, lambda x: 0, lambda x: 1)
print(f"∫∫ (x² + y²) dA over [0,1]² = {result:.6f}")

# Engineering application: Compute drag from pressure field
x = np.linspace(0, 2, 100)
pressure = 101325 + 5000 * np.exp(-x**2)  # Pa
drag = integrate.simpson(pressure, x=x)
print(f"\nDrag force (simplified): {drag:.2f} N")

## ODE Solvers

Model dynamic systems: structural dynamics, control systems, chemical reactions.

In [ ]:
# 3. ODE Solvers
print("\n=== ODE Solvers ===")

# Damped harmonic oscillator
def damped_oscillator(t, y):
    """y = [position, velocity]"""
    m, c, k = 1.0, 0.1, 4.0  # mass, damping, stiffness
    x, v = y
    dxdt = v
    dvdt = (-c * v - k * x) / m
    return [dxdt, dvdt]

t = np.linspace(0, 10, 1000)
y0 = [1.0, 0.0]  # initial position, velocity

sol = integrate.solve_ivp(damped_oscillator, [t[0], t[-1]], y0, t_eval=t)

plt.figure(figsize=(10, 6))
plt.plot(t, sol.y[0], 'b-', label='Position')
plt.plot(t, sol.y[1], 'r--', label='Velocity')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.title('Damped Harmonic Oscillator')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final position: {sol.y[0, -1]:.4f}")
print(f"Final velocity: {sol.y[1, -1]:.4f}")

## Interpolation

Interpolate CFD data between grid points.

In [ ]:
# 4. Interpolation
print("\n=== Interpolation ===")

# 1D interpolation
x = np.array([0, 1, 2, 3, 4])
y = np.array([0, 1, 4, 9, 16])

f_linear = interpolate.interp1d(x, y, kind='linear')
f_cubic = interpolate.interp1d(x, y, kind='cubic')

x_new = np.linspace(0, 4, 100)

plt.figure(figsize=(10, 6))
plt.plot(x, y, 'ko', label='Data points')
plt.plot(x_new, f_linear(x_new), 'b-', label='Linear')
plt.plot(x_new, f_cubic(x_new), 'r--', label='Cubic')
plt.xlabel('x')
plt.ylabel('y')
plt.title('1D Interpolation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 2D interpolation (CFD field)
x = np.linspace(0, 1, 10)
y = np.linspace(0, 1, 10)
X, Y = np.meshgrid(x, y)
Z = np.sin(np.pi * X) * np.cos(np.pi * Y)

f_2d = interpolate.RectBivariateSpline(x, y, Z)
x_new = np.linspace(0, 1, 50)
y_new = np.linspace(0, 1, 50)
X_new, Y_new = np.meshgrid(x_new, y_new)
Z_new = f_2d(X_new, Y_new)

plt.figure(figsize=(10, 6))
plt.contourf(X_new, Y_new, Z_new, levels=20, cmap='viridis')
plt.colorbar(label='Interpolated Field')
plt.title('2D Interpolation of CFD Field')
plt.xlabel('X')
plt.ylabel('Y')
plt.tight_layout()
plt.show()

## Exercises

1. Optimize a 3-parameter model to fit experimental data.
2. Solve a system of ODEs modeling a chemical reactor.
3. Interpolate a 3D CFD field and compute integrals over subdomains.